# AirLLM Local Lab — Results Analysis

This notebook loads the committed benchmark results from `results/` and reproduces
the key tables and charts that appear in the README.  It is a companion to `uv run report`.

**Prerequisites:** `uv sync` must have been run once; all `results/*.json` must exist
(run the pipeline phases first if they are missing).

In [ ]:
import json
from pathlib import Path

ROOT = Path().resolve().parent
RESULTS = ROOT / "results"

summary = json.loads((RESULTS / "benchmark_summary.json").read_text())
raw = json.loads((RESULTS / "benchmark_raw.json").read_text())
economics = json.loads((RESULTS / "economics.json").read_text())
e1 = json.loads((RESULTS / "extension_e1.json").read_text())
e3 = json.loads((RESULTS / "extension_e3.json").read_text())

print("Loaded results OK")

## 1. Benchmark Summary (precision sweep)

In [ ]:
import pandas as pd

ok_rows = [r for r in summary if isinstance(r, dict) and r.get("status") == "ok"]
df = pd.DataFrame(ok_rows)
display(df[["precision", "ttft_s", "throughput_tps", "peak_ram_mb", "energy_j", "quality_normalised"]])

## 2. Raw Repetition Data (cold vs warm cache)

In [ ]:
import statistics

raw_df = pd.DataFrame(raw)
display(raw_df)


ttfts = raw_df["ttft_s"].tolist()
print(f"TTFT  median={statistics.median(ttfts):.3f}s  IQR={max(ttfts) - min(ttfts):.3f}s")

## 3. Extension E1 — I/O Location Sensitivity

In [ ]:
import matplotlib.pyplot as plt

locations = ["Internal NVMe\n(~/airllm_cache)", "/tmp\n(same drive)"]
times = [e1["internal_ssd"], e1["tmp"]]

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(locations, times, color=["steelblue", "coral"])
ax.bar_label(bars, fmt="%.3f s", padding=3)
ax.set_ylabel("TTFT (s)")
ax.set_title("Extension E1: Shard-location I/O sensitivity")
ax.set_ylim(0, max(times) * 1.2)
speedup = (e1["tmp"] - e1["internal_ssd"]) / e1["tmp"] * 100
ax.text(0.5, 0.92, f"NVMe is {speedup:.1f}% faster", ha="center", transform=ax.transAxes, fontsize=9)
plt.tight_layout()
plt.show()

## 4. Extension E3 — Page-Cache Warmup Curve

In [ ]:
cold_times = e3["cold"]
warm_times = e3["warm"]
all_times = cold_times + warm_times
labels = ["cold"] + [f"warm {i}" for i in range(1, len(warm_times) + 1)]
colors = ["coral"] + ["steelblue"] * len(warm_times)

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(labels, all_times, color=colors)
ax.bar_label(bars, fmt="%.2f s", padding=3)
ax.set_ylabel("TTFT (s)")
ax.set_title(f"Extension E3: cold→warm speedup {e3['speedup']:.2f}×")
ax.axhline(
    y=statistics.mean(warm_times), color="green", linestyle="--", label=f"warm avg {statistics.mean(warm_times):.2f} s"
)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Economic Break-Even Analysis

In [ ]:
import numpy as np

volumes = np.linspace(0, 150_000_000, 300)

# On-prem: fixed = $65.53/month, variable ≈ 0
onprem = economics.get("onprem_fixed_monthly", 65.53) * np.ones_like(volumes)

# API: Claude 3 Haiku $0.00025/1k input + $0.00125/1k output
price_per_1k = 0.4 * 0.00025 + 0.6 * 0.00125  # blended for 40% input / 60% output
api_cost = volumes / 1000 * price_per_1k

crossover = economics.get("crossover_tokens", 79_580_000)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(volumes / 1e6, onprem, label="On-Prem (fixed)", color="steelblue")
ax.plot(volumes / 1e6, api_cost, label="Managed API (Claude 3 Haiku)", color="coral")
ax.axvline(x=crossover / 1e6, color="green", linestyle="--", label=f"Break-even {crossover / 1e6:.1f}M tok/mo")
ax.set_xlabel("Monthly token volume (millions)")
ax.set_ylabel("Monthly cost (USD)")
ax.set_title("On-Prem vs Managed API break-even")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Break-even: {crossover:,.0f} tokens/month = {crossover / 1e6:.1f}M")

## 6. Roofline Model Summary

| Metric | Value |
|---|---|
| Total shard data | 2.0 GB (22 layers × ~93 MB) |
| Generation time (NVMe) | 33.793 s |
| Effective bandwidth | 60.6 MB/s |
| NVMe peak (M3 Pro) | ~7,000 MB/s |
| NVMe utilisation | **0.9%** |
| I/O-to-compute ratio | **9,000×** |

AirLLM is I/O-bound, not compute-bound.  The bottleneck is Python/mmap overhead per layer, not raw disk bandwidth.